# Capítulo 4 — Contar cosas: Poisson, ruido y fluctuaciones

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## ¿Cuántas cuentas hacen falta para poder decir «hemos detectado algo»?

Dibuja la significancia s/sqrt(b) frente al tiempo de medida para varias
relaciones señal/fondo, y el mínimo tiempo necesario para llegar a 5 sigma.

La figura responde: ¿por qué medir el doble de tiempo no duplica la
significancia?

Ejecutar:  python fig_deteccion_fondo.py

*(script original: `codigo/fig_deteccion_fondo.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

TASA_FONDO = 100.0                       # cuentas por hora
COCIENTES = [0.30, 0.10, 0.03, 0.01]     # tasa de señal / tasa de fondo
t = np.logspace(-1, 4, 400)              # horas

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.1))

for cociente, color in zip(COCIENTES, [C.blue, C.green, C.ochre, C.red]):
    s = cociente * TASA_FONDO * t
    b = TASA_FONDO * t
    significancia = s / np.sqrt(b)
    ax1.loglog(t, significancia, color=color, lw=1.9,
               label=f"$s/b$ = {cociente:.2f}")
    t5 = 25 / (cociente**2 * TASA_FONDO)
    ax1.plot(t5, 5, "o", color=color, ms=6)

ax1.axhline(5, color=C.ink, ls="--", lw=1.2)
ax1.text(0.12, 5.5, "umbral de descubrimiento, 5$\\sigma$", fontsize=8.4,
         color=C.ink)
ax1.axhline(3, color=C.grey, ls=":", lw=1.0)
ax1.text(0.12, 3.1, "«indicio», 3$\\sigma$", fontsize=8, color=C.grey)
ax1.set_xlabel("tiempo de medida (h)")
ax1.set_ylabel(r"significancia $s/\sqrt{b}$")
ax1.set_title(r"La significancia crece como $\sqrt{t}$")
ax1.legend(fontsize=8, loc="lower right")

# --- Coste: horas necesarias para 5 sigma --------------------------------
cocientes = np.logspace(-3, 0, 200)
horas = 25 / (cocientes**2 * TASA_FONDO)
ax2.loglog(cocientes, horas, color=C.red, lw=2.0)
for c, etiqueta in [(0.3, "señal fuerte"), (0.03, "señal débil"),
                    (0.003, "señal muy débil")]:
    h = 25 / (c**2 * TASA_FONDO)
    ax2.plot(c, h, "o", color=C.ink, ms=5)
    txt = f"{h:.0f} h" if h < 8760 else f"{h/8760:.0f} años"
    ax2.annotate(f"{etiqueta}\n{txt}", (c, h), textcoords="offset points",
                 xytext=(-6, 8), fontsize=8, ha="right")
ax2.set_xlabel("cociente señal/fondo  $s/b$")
ax2.set_ylabel("horas de medida para 5$\\sigma$")
ax2.set_title("Dividir la señal por 10 multiplica el tiempo por 100")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué una foto con poca luz sale con grano?

Simula la misma imagen recogida con números crecientes de fotones. La figura
responde: ¿cómo se ve, literalmente, que el ruido relativo baja como
1/sqrt(N)?

Ejecutar:  python fig_imagen_fotones.py

*(script original: `codigo/fig_imagen_fotones.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(4)

# Escena sintética: dos discos y un gradiente suave
n_pix = 120
y, x = np.mgrid[0:n_pix, 0:n_pix] / n_pix
escena = 0.25 + 0.35 * x
escena += 0.45 * (((x - 0.32) ** 2 + (y - 0.62) ** 2) < 0.022)
escena += 0.30 * (((x - 0.70) ** 2 + (y - 0.35) ** 2) < 0.010)
escena /= escena.max()

FOTONES = [1, 10, 100, 1000, 10_000, 100_000]

fig, axes = plt.subplots(2, 3, figsize=(10.4, 6.4))
for ax, n_medio in zip(axes.ravel(), FOTONES):
    imagen = r.poisson(escena * n_medio)
    ax.imshow(imagen, cmap="gray", interpolation="nearest")
    ax.set_xticks([]), ax.set_yticks([]), ax.grid(False)
    snr = np.sqrt(n_medio)
    ax.set_title(f"$\\langle N\\rangle$ = {n_medio:,} fotones/píxel\n"
                 f"ruido relativo $\\approx$ {100/snr:.1f} %".replace(",", " "),
                 fontsize=9.5)

fig.suptitle("La misma escena, seis exposiciones: el grano es Poisson, no el "
             "sensor", fontsize=11.5, y=0.99)
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Si los autobuses pasan cada 10 minutos, ¿cuánto esperas de media?

Simula llegadas de Poisson y mide (a) el intervalo medio entre autobuses y
(b) el intervalo en el que cae un pasajero que llega en un instante al azar.

La figura responde: ¿por qué esperas más de lo que dice el horario, incluso si
el horario es honesto?

Ejecutar:  python fig_paradoja_autobus.py

*(script original: `codigo/fig_paradoja_autobus.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(17)

TASA = 1 / 10.0            # un autobús cada 10 minutos de media
T_TOTAL = 200_000.0

# Instantes de llegada de un proceso de Poisson
huecos = r.exponential(1 / TASA, int(T_TOTAL * TASA * 1.2))
llegadas = np.cumsum(huecos)
llegadas = llegadas[llegadas < T_TOTAL]
huecos = np.diff(llegadas)

# Un millón de pasajeros llegan en instantes uniformes
pasajeros = np.sort(r.uniform(0, llegadas[-1], 1_000_000))
idx = np.searchsorted(llegadas, pasajeros) - 1
valido = (idx >= 0) & (idx < len(huecos))
hueco_visto = huecos[idx[valido]]
espera = llegadas[idx[valido] + 1] - pasajeros[valido]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.0))

ax1.hist(huecos, bins=100, density=True, color=C.blue, alpha=0.55,
         edgecolor="none", label=f"huecos reales (media {huecos.mean():.1f} min)")
ax1.hist(hueco_visto, bins=100, density=True, histtype="step", lw=1.8,
         color=C.red,
         label=f"hueco que ve el pasajero (media {hueco_visto.mean():.1f} min)")
ax1.set_xlim(0, 60)
ax1.set_xlabel("duración del intervalo entre autobuses (min)")
ax1.set_ylabel("densidad")
ax1.set_title("El pasajero no ve un hueco cualquiera:\nve uno grande, porque "
              "son más anchos", fontsize=10)
ax1.legend(fontsize=8)

ax2.hist(espera, bins=100, density=True, color=C.ochre, alpha=0.6,
         edgecolor="none")
ax2.axvline(espera.mean(), color=C.ink, lw=1.8)
ax2.text(espera.mean() * 1.05, ax2.get_ylim()[1] * 0.8,
         f"espera media = {espera.mean():.1f} min\n"
         f"(el horario dice 10; la mitad serían 5)",
         fontsize=8.8, color=C.ink)
ax2.set_xlim(0, 50)
ax2.set_xlabel("tiempo de espera del pasajero (min)")
ax2.set_ylabel("densidad")
ax2.set_title("Y por eso espera 10 minutos, no 5", fontsize=10)

print(f"hueco medio real:     {huecos.mean():.2f} min")
print(f"hueco medio visto:    {hueco_visto.mean():.2f} min")
print(f"espera media:         {espera.mean():.2f} min")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Se cumple Poisson con datos reales de hace un siglo?

Compara dos conjuntos de datos históricos con la ley de Poisson: los conteos
de partículas alfa de Rutherford, Geiger y Bateman (1910) y las muertes por
coz de caballo en el ejército prusiano de Bortkiewicz (1898).

La figura responde: ¿es Poisson una idealización o describe datos reales?

Ejecutar:  python fig_poisson_datos.py

*(script original: `codigo/fig_poisson_datos.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# --- Rutherford, Geiger y Bateman (1910): 2608 intervalos de 7,5 s --------
alfa_k = np.arange(0, 15)
alfa_n = np.array([57, 203, 383, 525, 532, 408, 273, 139, 45, 27, 10, 4, 0, 1, 1])

# --- Bortkiewicz (1898): 200 cuerpo-años del ejército prusiano ------------
coz_k = np.arange(0, 5)
coz_n = np.array([109, 65, 22, 3, 1])

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))

for ax, k, n, titulo, unidad in [
    (axes[0], alfa_k, alfa_n, "Partículas $\\alpha$ en 7,5 s\nRutherford, Geiger y "
     "Bateman (1910)", "intervalos"),
    (axes[1], coz_k, coz_n, "Muertes por coz de caballo\nBortkiewicz (1898)",
     "cuerpo-años"),
]:
    total = n.sum()
    media = (k * n).sum() / total
    varianza = ((k - media) ** 2 * n).sum() / total
    esperado = total * stats.poisson.pmf(k, media)

    ax.bar(k, n, color=C.blue, alpha=0.55, width=0.75, label="observado")
    ax.plot(k, esperado, "o-", color=C.red, ms=5, lw=1.4,
            label=f"Poisson($\\lambda$={media:.2f})")
    ax.set_xlabel("número de sucesos en el intervalo")
    ax.set_ylabel(f"número de {unidad}")
    ax.set_title(titulo, fontsize=10)
    ax.legend()
    ax.text(0.97, 0.62,
            f"media = {media:.3f}\nvarianza = {varianza:.3f}\n"
            f"cociente = {varianza/media:.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8.8,
            color=C.ink,
            bbox=dict(boxstyle="round,pad=0.4", fc=C.light, ec=C.grey, lw=0.6))
    print(f"{titulo.splitlines()[0]}: media={media:.3f} var={varianza:.3f} "
          f"var/media={varianza/media:.3f}")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
